In [172]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [173]:
model = "Llama-3-8B-Instruct_3_shot"

p2_label_path = "chia_label/p2"
ready_path = f"model_output/{model}/ready"
failed_model_path = f"model_output/{model}/failed_inner"
p2_model_formatted_path = f"model_output/{model}/formatted"

eval_path = f"evaluate/batch1"

In [174]:
def extract_nct_number(filename):
    parts = filename.split('_')
    nct_number = None
    file_type = None
    for part in parts:
        if part.startswith("NCT"):
            nct_number = part
        if part in ["inc", "exc"]:
            file_type = part
    return nct_number+"_"+file_type

def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)


def extract_logical_structure(data):
    structure = defaultdict(int)
    def traverse(node, depth=0):
        nonlocal structure
        structure["depth"] = max(structure["depth"], depth)

        if isinstance(node, dict):
            for key in node:
                if key in ["AND", "OR", "NOT"]:
                    structure[key] += 1
                traverse(node[key], depth + 1)
        elif isinstance(node, list):
            for item in node:
                traverse(item, depth + 1)

    traverse(data)
    return dict(structure)

In [175]:
label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}

common_ncts = set(label_files.keys()).intersection(model_files.keys())

In [176]:
labels = []
predictions = []
success_data = []

In [177]:
for nct in common_ncts:# ["NCT00050349_exc"]: #
    try:
        label_data = read_json(label_files[nct])
        model_data = read_json(model_files[nct])

        label_structure = extract_logical_structure(label_data)
        model_structure = extract_logical_structure(model_data)
        success_data.append({
            'NCT': nct,
            'label_AND': label_structure.get('AND', 0),
            'label_OR': label_structure.get('OR', 0),
            'label_NOT': label_structure.get('NOT', 0),
            'label_DEPTH': label_structure.get('depth', 0),
            'model_AND': model_structure.get('AND', 0),
            'model_OR': model_structure.get('OR', 0),
            'model_NOT': model_structure.get('NOT', 0),
            'model_DEPTH': model_structure.get('depth', 0)
        })

        labels.append(label_structure)
        predictions.append(model_structure)
        #shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
    except Exception as e:
        print(f"Error processing NCT {nct}: {e}")
        #shutil.copy(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))

Error processing NCT NCT02015923_exc: Expecting ',' delimiter: line 73 column 57 (char 2024)
Error processing NCT NCT02567214_inc: Expecting property name enclosed in double quotes: line 31 column 17 (char 837)
Error processing NCT NCT01994382_inc: Extra data: line 117 column 1 (char 5136)
Error processing NCT NCT03513757_exc: Extra data: line 58 column 1 (char 1338)
Error processing NCT NCT01967420_inc: Extra data: line 65 column 1 (char 1657)
Error processing NCT NCT03518034_exc: Expecting property name enclosed in double quotes: line 30 column 15 (char 902)
Error processing NCT NCT02527512_exc: Extra data: line 17 column 2 (char 274)
Error processing NCT NCT02632760_exc: Extra data: line 89 column 5 (char 2455)
Error processing NCT NCT03171987_inc: Extra data: line 68 column 1 (char 2325)
Error processing NCT NCT02509091_exc: Extra data: line 77 column 1 (char 2031)
Error processing NCT NCT02882113_inc: Extra data: line 74 column 1 (char 1858)
Error processing NCT NCT01809041_inc: E

In [178]:
df_success = pd.DataFrame(success_data).set_index('NCT')
df_success.to_csv(eval_path+f'/{model}_eval.csv')

In [179]:
true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values

metrics = {}
for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
    y_true = true_values[:, i]
    y_pred = predicted_values[:, i]

    metrics[metric] = {
        'accuracy': round(accuracy_score(y_true, y_pred), 3),
        'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3)
       # 'confusion_matrix': confusion_matrix(y_true, y_pred)
    }
    metrics_df = pd.DataFrame(metrics).T  
    num_nct_files = len(df_success)
    metrics_df['num_nct_files'] = num_nct_files
    metrics_df['model_name'] = model
    # Save Metrics to CSV
    #metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))
    

for metric, values in metrics.items():
    print(f"Metrics for {metric}:")
    print(f"  Accuracy: {values['accuracy']}")
    print(f"  Precision: {values['precision']}")
    print(f"  Recall: {values['recall']}")
    print(f"  F1 Score: {values['f1_score']}")
    #print(f"  Confusion Matrix:\n{values['confusion_matrix']}\n")

Metrics for AND:
  Accuracy: 0.124
  Precision: 0.112
  Recall: 0.124
  F1 Score: 0.111
Metrics for OR:
  Accuracy: 0.281
  Precision: 0.332
  Recall: 0.281
  F1 Score: 0.298
Metrics for NOT:
  Accuracy: 0.727
  Precision: 0.693
  Recall: 0.727
  F1 Score: 0.709
Metrics for DEPTH:
  Accuracy: 0.2
  Precision: 0.208
  Recall: 0.2
  F1 Score: 0.198


In [180]:
matching_rows = df_success[df_success['label_AND'] == df_success['model_AND']][['label_AND', 'model_AND']]
matching_rows

,label_AND,model_AND
NCT,,
NCT02954029_inc,2,2
NCT02707874_inc,6,6
NCT00586898_exc,5,5
NCT02092467_inc,3,3
NCT00250640_exc,2,2
...,...,...
NCT02481518_inc,5,5
NCT02344888_exc,7,7
NCT02862314_inc,5,5
